<a href="https://colab.research.google.com/github/Jalilnkh/PyTorch-with-Examples-2024/blob/main/train_en_azb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Kartal-Ol/en-azb-548k")


In [ ]:
from transformers import AutoTokenizer

# Load the Arabic tokenizer
tokenizer = AutoTokenizer.from_pretrained("asafaya/bert-base-arabic")

In [ ]:
# Tokenize function for batched tokenization
def tokenize_function(examples):
    # Extract 'en' and 'azb' for each example in the batch
    source_texts = [example['en'] for example in examples['translation']]
    target_texts = [example['azb'] for example in examples['translation']]

    # Tokenize both source (English) and target (Azerbaijani Arabic script)
    source = tokenizer(source_texts, padding="max_length", truncation=True, max_length=128)
    target = tokenizer(target_texts, padding="max_length", truncation=True, max_length=128)

    return {
        'input_ids': source['input_ids'],
        'attention_mask': source['attention_mask'],
        'labels': target['input_ids']
    }

# Tokenize the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [ ]:
from transformers import T5ForConditionalGeneration

# Load T5 model
model = T5ForConditionalGeneration.from_pretrained("t5-small")


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",          # Where to save model checkpoints
    evaluation_strategy="epoch",    # Evaluate at the end of every epoch
    learning_rate=2e-5,             # Learning rate
    per_device_train_batch_size=16, # Batch size per GPU
    per_device_eval_batch_size=16,  # Batch size for validation
    num_train_epochs=3,             # Number of epochs
    weight_decay=0.01,              # Weight decay for regularization
    save_total_limit=3              # Keep only the last 3 checkpoints
)